---
## 7. Evaluation — Precision@k / Recall@k
**Work Package: Performance Evaluation**

**Method 1:** Rank one held-out positive against 99 random negatives (repeated
per positive, up to 3 per user)  
**Method 2:** Leave-one-out — hold out the user's most recent interaction

Both methods evaluate SVD and BPR identically for a fair comparison.
BPR is then tuned with Optuna to confirm whether tuning closes any gap
before the final model is selected.


In [ ]:
def prec_at_k(rec, rel, k): return len(set(rec[:k]) & rel) / k if k else 0
def rec_at_k(rec, rel, k):  return len(set(rec[:k]) & rel) / len(rel) if rel else 0
def f1(p, r): return 2*p*r/(p+r) if p+r else 0

K_VALUES = [1, 3, 5, 10]

train_by_user = {str(k):{str(r) for r in v}
                  for k,v in df_train.groupby('user_id')['recipe_id'].apply(set).items()}
test_by_user  = {str(k):{str(r) for r in v}
                  for k,v in df_test.groupby('user_id')['recipe_id'].apply(set).items()}
known_rids = set(R2I.keys())

eval_users = []
eval_test  = {}
for u in test_by_user:
    if u not in train_by_user or u not in U2I: continue
    known_pos = {r for r in test_by_user[u] if r in known_rids}
    if known_pos:
        eval_users.append(u)
        eval_test[u] = known_pos

active_users = [u for u in eval_users if len(train_by_user.get(u,set())) >= 10]
print(f'Eval users: {len(eval_users):,}  |  Active users: {len(active_users):,}')
validate(len(active_users) > 100, 'Sufficient active users', '> 100', f'{len(active_users):,}')


In [ ]:
# ── Reusable evaluation functions ──────────────────────────────────────────────
def eval_method1(score_fn, label, max_users=500):
    prec = {k: [] for k in K_VALUES}
    rec  = {k: [] for k in K_VALUES}
    for uid in active_users[:max_users]:
        seen = train_by_user.get(uid, set())
        all_pos = eval_test[uid]
        for pos in list(all_pos)[:3]:
            neg_pool = list(known_rids - seen - all_pos)
            if len(neg_pool) < 99: continue
            candidates = [pos] + random.sample(neg_pool, 99)
            preds = sorted([(r, score_fn(uid, r)) for r in candidates],
                            key=lambda x: x[1], reverse=True)
            rec_ids = [p[0] for p in preds]
            rel = {pos}
            for k in K_VALUES:
                prec[k].append(prec_at_k(rec_ids, rel, k))
                rec[k].append(rec_at_k(rec_ids, rel, k))
    print(f'{label}:')
    print(f'{"k":<5}{"Precision":>12}{"Recall":>12}')
    for k in K_VALUES:
        print(f'{k:<5}{np.mean(prec[k]):>12.4f}{np.mean(rec[k]):>12.4f}')
    return prec, rec

def eval_method2(score_fn, label, max_users=500):
    prec = {k: [] for k in K_VALUES}
    rec  = {k: [] for k in K_VALUES}
    for uid in active_users[:max_users]:
        seen = train_by_user.get(uid, set())
        pos_list = list(eval_test.get(uid, set()))
        if not pos_list: continue
        held = pos_list[0]
        neg_pool = list(known_rids - seen - {held})
        if len(neg_pool) < 99: continue
        candidates = [held] + random.sample(neg_pool, 99)
        preds = sorted([(r, score_fn(uid, r)) for r in candidates],
                        key=lambda x: x[1], reverse=True)
        rec_ids = [p[0] for p in preds]
        rel = {held}
        for k in K_VALUES:
            prec[k].append(prec_at_k(rec_ids, rel, k))
            rec[k].append(rec_at_k(rec_ids, rel, k))
    print(f'{label}:')
    for k in K_VALUES:
        print(f'  k={k:<3} Precision={np.mean(prec[k]):.4f}  Recall={np.mean(rec[k]):.4f}')
    return prec, rec


In [ ]:
# ── Evaluate SVD and BPR (default) — both methods ──────────────────────────────
print('=== SVD — Method 1 ===')
svd_prec, svd_rec = eval_method1(
    lambda uid,rid: svd.predict(U2I[uid],R2I[rid]).est, 'SVD')

print('\n=== BPR (default) — Method 1 ===')
bpr_prec, bpr_rec = eval_method1(bpr_score, 'BPR (default)')

validate(np.mean(svd_prec[1]) > 0, 'SVD Precision@1 non-zero', '> 0', f'{np.mean(svd_prec[1]):.4f}')
validate(np.mean(bpr_prec[1]) > 0, 'BPR Precision@1 non-zero', '> 0', f'{np.mean(bpr_prec[1]):.4f}')

print('\n=== SVD — Method 2 (LOO) ===')
svd_prec2, svd_rec2 = eval_method2(
    lambda uid,rid: svd.predict(U2I[uid],R2I[rid]).est, 'SVD')

print('\n=== BPR (default) — Method 2 (LOO) ===')
bpr_prec2, bpr_rec2 = eval_method2(bpr_score, 'BPR (default)')


In [ ]:
# ── Tune BPR — give it a fair chance before final model selection ─────────────
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

TUNE_USERS_BPR = active_users[:100]

def bpr_objective(trial):
    kf    = trial.suggest_int('k', 16, 128)
    lr    = trial.suggest_float('learning_rate', 0.001, 0.1, log=True)
    reg   = trial.suggest_float('lambda_reg', 0.0001, 0.05, log=True)
    iters = trial.suggest_int('max_iter', 50, 200)

    m = BPR(k=kf, max_iter=iters, learning_rate=lr,
             lambda_reg=reg, seed=42, verbose=False)
    m.fit(cornac_data)

    hits = []
    for uid in TUNE_USERS_BPR:
        seen = train_by_user.get(uid, set())
        pos_list = list(eval_test.get(uid, set()))
        if not pos_list: continue
        pos = pos_list[0]
        neg_pool = list(known_rids - seen - eval_test.get(uid, set()))
        if len(neg_pool) < 99: continue
        cands = [pos] + random.sample(neg_pool, 99)
        try:
            preds = sorted(
                [(r, m.score(cornac_data.uid_map[uid], cornac_data.iid_map[r]))
                 for r in cands if r in cornac_data.iid_map],
                key=lambda x: x[1], reverse=True
            )
            rank = [p[0] for p in preds].index(pos) + 1
            hits.append(int(rank <= 10))
        except:
            continue
    return np.mean(hits) if hits else 0.0

print('Tuning BPR (30 trials)...')
bpr_study = optuna.create_study(direction='maximize',
                                  sampler=optuna.samplers.TPESampler(seed=42))
bpr_study.optimize(bpr_objective, n_trials=30)

bpr_best = bpr_study.best_params
print(f'\nBest BPR Hit@10 (tuning): {bpr_study.best_value:.1%}')
for k, v in bpr_best.items(): print(f'  {k:<15}={v}')


In [ ]:
# ── Retrain BPR with tuned hyperparameters, re-evaluate ────────────────────────
print('Retraining BPR with tuned hyperparameters...')
bpr_tuned = BPR(k=bpr_best['k'], max_iter=bpr_best['max_iter'],
                  learning_rate=bpr_best['learning_rate'],
                  lambda_reg=bpr_best['lambda_reg'], seed=42, verbose=True)
bpr_tuned.fit(cornac_data)

def bpr_tuned_score(uid_str, rid_str):
    try:
        return bpr_tuned.score(cornac_data.uid_map[uid_str], cornac_data.iid_map[rid_str])
    except KeyError:
        return -999.0

with open('models/bpr_tuned.pkl','wb') as f:
    pickle.dump({'model':bpr_tuned,'params':bpr_best},f)

print('\n=== BPR (tuned) — Method 1 ===')
bpr_tuned_prec, bpr_tuned_rec = eval_method1(bpr_tuned_score, 'BPR (tuned)')


In [ ]:
# ── Final three-way comparison and model selection ─────────────────────────────
print('=== Final Comparison: SVD vs BPR (default) vs BPR (tuned) ===')
print(f'{"k":<5}{"SVD":>10}{"BPR-default":>14}{"BPR-tuned":>12}{"Winner":>14}')
for k in K_VALUES:
    s  = np.mean(svd_prec[k])
    bd = np.mean(bpr_prec[k])
    bt = np.mean(bpr_tuned_prec[k])
    winner = max([('SVD',s), ('BPR-default',bd), ('BPR-tuned',bt)], key=lambda x: x[1])[0]
    print(f'{k:<5}{s:>10.4f}{bd:>14.4f}{bt:>12.4f}{winner:>14}')

all_candidates = {
    'svd':         np.mean(svd_prec[1]),
    'bpr_default': np.mean(bpr_prec[1]),
    'bpr_tuned':   np.mean(bpr_tuned_prec[1]),
}
winner_name = max(all_candidates, key=all_candidates.get)
print(f'\n🏆 Overall winner: {winner_name}  ({all_candidates[winner_name]:.1%})')

ACTIVE_CF_MODEL = 'svd' if winner_name == 'svd' else 'bpr'
if winner_name == 'bpr_tuned':
    bpr = bpr_tuned
    bpr_score = bpr_tuned_score

validate(all_candidates[winner_name] > 0.10,
         'Winning model beats random baseline (k=1)',
         '> 10% (random baseline)',
         f'{all_candidates[winner_name]:.1%}')


In [ ]:
# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(K_VALUES, [np.mean(svd_prec[k]) for k in K_VALUES],
         marker='o', color=C_PURPLE, linewidth=2, markersize=7, label='SVD')
ax.plot(K_VALUES, [np.mean(bpr_prec[k]) for k in K_VALUES],
         marker='^', color=C_FLAG, linewidth=2, markersize=7, label='BPR (default)')
ax.plot(K_VALUES, [np.mean(bpr_tuned_prec[k]) for k in K_VALUES],
         marker='D', color=C_AFTER, linewidth=2, markersize=7, label='BPR (tuned)')
ax.plot(K_VALUES, [k/100 for k in K_VALUES],
         marker='s', color='gray', linestyle='--', markersize=5, label='Random')
ax.set_xlabel('k'); ax.set_ylabel('Precision@k')
ax.set_title('Model Comparison — Method 1', fontweight='bold')
ax.legend(fontsize=8)

ax = axes[1]
ax.plot(K_VALUES, [np.mean(svd_prec2[k]) for k in K_VALUES],
         marker='o', color=C_PURPLE, linewidth=2, markersize=7, label='SVD')
ax.plot(K_VALUES, [np.mean(bpr_prec2[k]) for k in K_VALUES],
         marker='^', color=C_FLAG, linewidth=2, markersize=7, label='BPR (default)')
ax.plot(K_VALUES, [k/100 for k in K_VALUES],
         marker='s', color='gray', linestyle='--', markersize=5, label='Random')
ax.set_xlabel('k'); ax.set_ylabel('Precision@k')
ax.set_title('Model Comparison — Method 2 (LOO)', fontweight='bold')
ax.legend(fontsize=8)

plt.suptitle('SVD vs BPR — Precision@k Across Methods', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/svd_vs_bpr_tuned.png', dpi=120, bbox_inches='tight')
plt.show()
